# EDA Cyber Splits

Split analysis for cyber datasets (UNSW-NB15).

Steps:
- Inspect train/test files and label balance.
- Create a validation split from training data.
- Summarize class balance across splits.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

cyber_root = REPO_ROOT / 'data' / 'raw' / 'cyber'
train_path = cyber_root / 'UNSW_NB15_training-set.csv'
test_path = cyber_root / 'UNSW_NB15_testing-set.csv'

summary = {
    'train': {},
    'test': {},
    'val': {},
}

label_candidates = ['label', 'Label', 'is_attack', 'target']

def summarize_split(df, name):
    summary[name]['rows'] = int(df.shape[0])
    summary[name]['cols'] = int(df.shape[1])
    label_col = next((col for col in label_candidates if col in df.columns), None)
    if label_col:
        counts = df[label_col].value_counts().to_dict()
        total = sum(counts.values())
        ratio = counts.get(1, 0) / total if total else 0
        summary[name]['label_col'] = label_col
        summary[name]['label_counts'] = counts
        summary[name]['attack_ratio'] = round(ratio, 6)
        print(f'{name} label distribution:', counts)
    else:
        print(f'No label column found for {name}')

if train_path.exists():
    train_df = pd.read_csv(train_path, low_memory=False)
    print('Train shape:', train_df.shape)
    summarize_split(train_df, 'train')
else:
    print('Missing train:', train_path)

if test_path.exists():
    test_df = pd.read_csv(test_path, low_memory=False)
    print('Test shape:', test_df.shape)
    summarize_split(test_df, 'test')
else:
    print('Missing test:', test_path)

if train_path.exists():
    label_col = summary['train'].get('label_col')
    if label_col:
        train_split, val_split = train_test_split(
            train_df, test_size=0.2, random_state=42, stratify=train_df[label_col]
        )
        print('Val shape:', val_split.shape)
        summarize_split(val_split, 'val')


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_cyber_splits_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize cyber-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'cyber' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No cyber entries found in TRAINING_DATA.json')
    else:
        print('cyber datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
